# Peak-detection diagnostics (visual proof)

For one run and one forced column: the relative density perturbation vs time at every altitude in the growth window (cells nearest 150 and 320 km, i.e. 148.3–315.4 km). The chosen per-cycle peaks are red dots, the last cycle (the value behind G_max) is ringed, and the dotted line marks the end of the forcing.

Three backgrounds are shown:
* `mad_column0`: root `optimized_loader`, azimuth column 0 only.
* `azimean_snapshot`: the 2024 recipe, azimuthal mean of the snapshot 2 outputs before forcing.
* `azimean_preforcing`: the refactored S1–S6 recipe, mean over all columns and the 17 pre-forcing outputs.

Peak detection is identical in all three. The dots sit on the cycle maxima every time. What changes is the vertical offset of each series, i.e. the reference density at that altitude. That offset is what made the S1–S3 growth profiles crooked.

This notebook loads the run's fields (GB-scale for S4–S6).

In [1]:
import matplotlib.pyplot as plt
import numpy as np

from dsmclib import plotting, registry
from dsmclib.constants import PADDING
from dsmclib.growth import GROWTH_PEAK_KWARGS, forced_columns
from dsmclib.paths import output_dir
from dsmclib.peaks import process_growth_num_amp

plotting.use_style(font_scale=1.0)
FIG_DIR = output_dir("s1_s6_azimuthal", "figures")

ALIAS = "S1"
ALT_RANGE = (150, 320)
run = registry.open_run(ALIAS)
run.release_macro()
cols = forced_columns(run)
COLUMNS = [int(cols[len(cols) // 2]), int(cols[0])]   # centre and edge of the forcing footprint
COLUMNS

Loading combined_10runs_macro_grids_data.pkl (0.46 GB)


[69, 63]

In [2]:
mask = run.forcing_time_mask(PADDING)
c0, c1 = (run.altitude_index(a, "nearest") for a in ALT_RANGE)
t, alt = run.params.time[mask], run.params.radial_cells[c0:c1 + 1]
BACKGROUNDS = ("mad_column0", "azimean_snapshot", "azimean_preforcing")

def peak_diagnostic(column, background):
    pert = run.density_perturbation(background=background)[..., column]
    growth, _, idx = process_growth_num_amp(pert, mask, run.params.radial_cells, c0, c1, run.params.num_periods,
                                            use_dynamic=True, return_indices=True, **GROWTH_PEAK_KWARGS)
    fig = plotting.plot_peak_diagnostics(pert[mask][:, c0:c1 + 1], t, alt, idx, wave_end=run.params.wave_end_time,
                                         title=f"{ALIAS} column {column}, background {background}: "
                                               f"last-cycle growth at {alt[-1]:.0f} km = {float(growth[-1, -1]):.2f}")
    fig.savefig(FIG_DIR / f"peaks_{ALIAS}_col{column}_{background}.png", dpi=150, bbox_inches="tight")
    return growth

In [3]:
growth = {}
for column in COLUMNS:
    for background in BACKGROUNDS:
        growth[column, background] = peak_diagnostic(column, background)

Background window selected: Start index = 0, End index = 2


/Volumes/MyBook/Education and Research/Research/DSMC Exosphere GWs/dsmc_analyzer/refactored/dsmclib/background.py:177: RuntimeWarning: divide by zero encountered in divide
  return (field - background[None, :, None]) / background[None, :, None]
/Volumes/MyBook/Education and Research/Research/DSMC Exosphere GWs/dsmc_analyzer/refactored/dsmclib/background.py:177: RuntimeWarning: invalid value encountered in divide
  return (field - background[None, :, None]) / background[None, :, None]


Background snapshot: index 15 (t = 5005.8 s), mean over 128 azimuth columns


Background: outputs 0-16 (t = 4907.8-5012.3 s), mean over time and 128 azimuth columns


## The reference profiles

Each background relative to `azimean_preforcing`, per altitude. A scatter of a few % is comparable to the perturbation amplitude itself near 150 km (≈0.12), so it rescales whole growth columns.

In [4]:
bgs = {b: run.background(method=b)["density"] for b in BACKGROUNDS}
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=True)
z = run.params.radial_cells
sel = z < 400
for b in ("mad_column0", "azimean_snapshot"):
    axes[0].plot(100 * (bgs[b][sel] / bgs["azimean_preforcing"][sel] - 1), z[sel], marker="o", ms=3, label=b)
axes[0].axvline(0, color="k", lw=0.8)
axes[0].axhspan(alt[0], alt[-1], color="C1", alpha=0.1, label="growth window")
axes[0].set(xlabel="background / azimean_preforcing − 1 (%)", ylabel="Altitude (km)")
axes[0].legend(fontsize=9)
for column in COLUMNS:
    for background, ls in zip(BACKGROUNDS, (":", "--", "-")):
        axes[1].plot(np.ma.filled(growth[column, background][:, -1].astype(float), np.nan), alt, ls=ls,
                     marker="o", ms=3, label=f"col {column}, {background}")
axes[1].set(xlabel="last-cycle growth")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / f"background_effect_{ALIAS}.png", dpi=150, bbox_inches="tight")